# Pima - entrainement v1 (diabete)

Conversion de ton `multivariable.py` en notebook, cellule par cellule.

**Ton code tel quel** - je n'ai rien ajoute ni corrige. L'etape 3 (standardisation) est signalee mais **pas ecrite** : c'est toujours la tienne.

In [18]:
! pip install scikit-learn 


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

## 1. Charger

In [3]:
df = pd.read_csv(r"D:\pima-predict\data\pima.csv")

## Regarder - c'est ici que tu visualises

Une expression par cellule pour que chaque sortie s'affiche. Regarde la ligne **`min`** de `describe()`.

In [4]:
df.shape

(768, 9)

In [5]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [6]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


## 2. Separer features / cible

In [7]:
y = df["Outcome"].to_numpy()                      # (767,)  les 0/1
X = df.drop(columns=["Outcome"]).to_numpy()       # (767, 8) les 8 features
feature_names = list(df.drop(columns=["Outcome"]).columns)   # l'ordre, pour le joblib
feature_names

['Pregnancies',
 'Glucose',
 'BloodPressure',
 'SkinThickness',
 'Insulin',
 'BMI',
 'DiabetesPedigreeFunction',
 'Age']

## 3. Split train / test

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)
X_train.shape, X_test.shape

((514, 8), (254, 8))

## [ROUGE] Etape 3 - Standardisation train-only (A TOI)

Tu as `train_mean` / `train_std` ci-dessous.

- La ligne `np.where(... isnan ...)` = **imputation** (remplir des NaN). Elle ne fait **rien** ici : il n'y a aucun NaN (les manquants sont des `0`).
- La **standardisation** `(X - mean) / std` n'est **toujours pas ecrite**. C'est ce qui manque, et c'est la cause de l'overflow.

Ecris la transformation (train ET test, avec les stats du TRAIN), et vire la ligne `np.where` morte.

In [9]:
cols_zero = [feature_names.index(c) for c in
             ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]]
for j in cols_zero:
    moyenne = X_train[X_train[:, j] != 0, j].mean()
    X_train[X_train[:, j] == 0, j] = moyenne
    X_test[ X_test[:, j] == 0, j]  = moyenne

# standardisation (stats du TRAIN)
train_mean = X_train.mean(axis=0)
train_std  = X_train.std(axis=0)
X_train = (X_train - train_mean) / train_std


## 4. Modele

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [11]:
w = np.zeros(8)  # Initialisation des poids pour huit caracteristiques
b = 0
lr = 0.05
nb_iteration = 10000

def train(X, y, w, b, lr, nb_iteration):
    for n in range(nb_iteration):
        z = X @ w + b
        y_pred = sigmoid(z)
        erreur = y_pred - y
        grad_w = X.T @ erreur / len(y)  # Gradient pour les poids
        grad_b = np.mean(erreur)

        w -= lr * grad_w
        b -= lr * grad_b

    print("Poids w1:", w[0])
    print("Poids w2:", w[1])
    print("Biais:", b)
    z = X @ w + b
    y_pred = sigmoid(z)

    resultat = np.where(y_pred > 0.5, 1, 0)
    print("Predictions:", resultat)
    print("Exactitude:", np.mean(resultat == y))
    return w, b

In [12]:


def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

## 5. Entrainer + evaluer

In [13]:
def predict(X, w, b):        

    for j in cols_zero:
        moyenne = X[X[:, j] != 0, j].mean()   # moyenne du train sans les 0
        X[X[:, j] == 0, j] = moyenne
    
    X  = (X  - train_mean) / train_std

    z = X @ w + b
    y_pred = sigmoid(z)
    return np.where(y_pred > 0.5, 1, 0)

w,b = train(X_train, y_train, w, b, lr, nb_iteration)
predictions = predict(X_test, w, b)
print("Exactitude sur le test:", accuracy(y_test, predictions))


Poids w1: 0.20116302504850483
Poids w2: 1.2079665925358096
Biais: -0.8643950994262686
Predictions: [0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 1 0 0 0 1 1 1 0 0 1 1 0 1 0
 0 1 0 0 0 0 1 0 0 0 1 1 0 0 0 0 0 0 0 0 1 1 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0
 1 0 0 0 0 0 0 0 1 1 0 0 1 0 0 1 1 1 0 0 0 0 0 0 1 1 0 0 0 0 0 1 0 1 1 0 0
 0 0 0 0 0 1 0 1 0 1 0 1 0 0 0 0 1 1 1 0 0 1 0 0 0 0 0 0 1 1 1 0 1 1 0 0 0
 0 0 0 0 0 1 0 0 0 0 0 0 0 1 1 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 1 1 1 0 0 0 1
 0 0 0 1 0 0 0 0 1 0 1 0 0 1 1 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 1 0
 1 0 0 0 0 0 0 1 0 1 1 0 0 1 0 1 0 1 0 0 1 0 0 0 1 0 0 1 1 0 1 1 1 0 0 0 1
 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 1 0 0 0 1 0 0 1 0 1 0 0 1 0 1 0 0 0 0
 0 0 1 0 0 0 0 0 1 1 0 0 0 1 1 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 1
 1 0 0 1 0 1 0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0
 0 1 0 0 0 0 1 0 1 1 0 0 0 0 0 1 1 0 0 0 0 1 1 0 1 1 0 0 0 1 0 0 0 0 0 0 1
 1 1 0 0 1 0 1 0 0 0 1 0 0 0 0 1 0 1 0 0 0 0 0 0 1 0 0 1 0 0 1 0 0 0 0 0 0
 

In [14]:
print("Exactitude sur le jeu de test:", accuracy(y_test, predictions))


Exactitude sur le jeu de test: 0.7480314960629921


In [15]:
import joblib
joblib.dump({"w" : w, "b": b,"mean":  train_mean, "std": train_std, "features": feature_names, "cols_zero": cols_zero}, r"D:\pima-predict\data\model.joblib")


['D:\\pima-predict\\data\\model.joblib']

In [ ]:
def predict_from_artifact(X, chemin):
    def sigmoid(z):
        return 1 / (1 + np.exp(-z))
    
    
    art = joblib.load(chemin)
    w = art["w"]        # les poids
    b = art["b"]        # le biais
    mean = art["mean"]     # mean du train (gelé)
    std = art["std"]      # std du train (gelé)
    features = art["features"] # l'ordre des colonnes
    cols_zero = art["cols_zero"] # les colonnes à traiter pour les 0
    
    for j in cols_zero:
        X[X[:, j] == 0, j] = mean[j]   # moyenne du train sans les 0

    X  = (X  - mean) / std

    z = X @ w + b
    y_pred = sigmoid(z)
    return np.where(y_pred > 0.5, 1, 0)

In [17]:
patient = np.array([3, 145, 72, 28, 130, 33.5, 0.52, 45])   # patient à prédire
predictions = predict_from_artifact(patient.reshape(1, -1), r"D:\pima-predict\data\model.joblib")
print("Prédiction pour le patient:", predictions)

Prédiction pour le patient: [1]
